# Spatial–temporal analysis
Frozen-model input-masking analysis, based on `draw_pic/fig3.ipynb`.
Compute increasing/decreasing temporal windows, local temporal occlusion and regional
occlusion on unchanged test galleries. Input occlusion describes model sensitivity;
it is not a causal or cortical-localization measurement. No retraining is performed here.
For multi-participant summaries, aggregate the exported tables by participant, not by trial.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from this notebook directory or the release root.
ROOT = Path.cwd() if (Path.cwd() / "brainae").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from figure_utils import *
style()
OUTPUT = ROOT / "figures"


In [ ]:
MODALITY='eeg'
DATA=ROOT/'data/prepared/eeg/sub-01'
CHECKPOINT=ROOT/'outputs/eeg/sub-01/best.pt'
PROTOTYPES=ROOT/'data/features/eeg_center.npy'
FEATURES=ROOT/'data/features/eeg_test.npy'
DEVICE='cpu'
PROTOTYPE_METRIC='original-dot'
SFREQ={'eeg':250,'meg':200,'mua':1000}[MODALITY]
T0=.005 if MODALITY=='meg' else 0.
WINDOW_SAMPLES=25
REGIONS={}  # Supply verified frontal/central/parietal/temporal/occipital channel lists.
RUN_CHANNEL_ANALYSIS=False  # One additional inference pass per channel.


In [ ]:
from brainae.runner import feature_array
measured=responses(DATA)
encoder,decoder=load_model(CHECKPOINT,MODALITY,DEVICE)
prototypes=np.load(PROTOTYPES,allow_pickle=False)
assert prototypes.shape==(len(measured),1024)
features=feature_array(FEATURES,np.load(DATA/'test_ids.npy'),assume_order=True)
predicted=infer(decoder,features,DEVICE)[:,0]
def score(x):
    return accuracy(infer(encoder,x[:,None],DEVICE),prototypes,PROTOTYPE_METRIC)
baseline=score(measured)
records=[]
for start in range(0,measured.shape[-1],WINDOW_SAMPLES):
    end=min(start+WINDOW_SAMPLES,measured.shape[-1])
    values={}
    for mode in ['increasing','decreasing','occlusion']:
        x=measured.copy()
        if mode=='increasing': x[:,:,end:]=0
        elif mode=='decreasing': x[:,:,:start]=0
        else: x[:,:,start:end]=0
        values[mode]=score(x)
    records.append({'start_s':T0+start/SFREQ,'end_s':T0+end/SFREQ,
                    'center_s':T0+(start+end-1)/(2*SFREQ),'baseline':baseline,
                    'encoding_r':np.nanmean(pearson_rows(measured[:,:,start:end],predicted[:,:,start:end])),**values})
temporal=pd.DataFrame(records)
OUTPUT.mkdir(parents=True,exist_ok=True)
temporal.to_csv(OUTPUT/'temporal_sensitivity.csv',index=False)


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(9,2.8),layout='constrained')
for ax,mode,title in zip(axes,['increasing','decreasing','occlusion'],['Increasing window','Decreasing window','Window occlusion']):
    x=temporal.end_s if mode=='increasing' else temporal.start_s if mode=='decreasing' else temporal.center_s
    y=100*(baseline-temporal[mode]) if mode=='occlusion' else 100*temporal[mode]
    ax.plot(x,y,'o-',color=BLUE,ms=3)
    ax.axhline(0 if mode=='occlusion' else 100*baseline,color=GRAY,ls='--',lw=1)
    ax.set(title=title,xlabel='Time (s)',ylabel='Top-1 decrease (pp)' if mode=='occlusion' else 'Top-1 accuracy (%)')
save(fig,OUTPUT,'temporal_sensitivity')

regions=dict(REGIONS)
if RUN_CHANNEL_ANALYSIS:
    regions.update({f'channel_{ch}':[ch] for ch in range(measured.shape[1])})
spatial=[]
for region,indices in regions.items():
    x=measured.copy(); x[:,indices,:]=0
    spatial.append({'region':region,'baseline':baseline,'accuracy':score(x),
                    'encoding_r':np.nanmean(pearson_rows(measured[:,indices],predicted[:,indices]))})
if spatial:
    spatial=pd.DataFrame(spatial); spatial['decrease_pp']=100*(baseline-spatial.accuracy)
    spatial.to_csv(OUTPUT/'spatial_sensitivity.csv',index=False)
    regional=spatial[~spatial.region.str.startswith('channel_')]
    if len(regional):
        fig,ax=plt.subplots(figsize=(5,3),layout='constrained')
        ax.bar(regional.region,regional.decrease_pp,color=BLUE)
        ax.axhline(0,color=GRAY,lw=.7); ax.tick_params(axis='x',rotation=30)
        ax.set(ylabel='Top-1 decrease (pp)',title='Regional occlusion')
        save(fig,OUTPUT,'regional_sensitivity')
else:
    print('Supply verified region indices to enable spatial panels.')

fig,ax=plt.subplots(figsize=(3.8,3),layout='constrained')
ax.scatter(100*(baseline-temporal.occlusion),temporal.encoding_r,color=BLUE,s=25)
ax.set(xlabel='Window occlusion: Top-1 decrease (pp)',ylabel='Window encoding Pearson r')
save(fig,OUTPUT,'encoding_decoding_temporal')
